In [ ]:
from dotenv import load_dotenv
from bs4 import BeautifulSoup  # MinerLoader 용
import re

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMyPDFLoader
from langchain_community.document_loaders import UnstructuredPDFLoader, PyPDFium2Loader
from langchain_community.document_loaders import PDFMinerLoader, PDFMinerPDFasHTMLLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader, PDFPlumberLoader

In [ ]:
load_dotenv()

In [ ]:
FILE_PATH = "C:/Users/grego/experiment/AI4CEO/data/SPRI_AI_Brief_2023년12월호.pdf"

In [ ]:
def show_metadata(docs):
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

PyPDF

In [ ]:
loader1 = PyPDFLoader(FILE_PATH)

docs1 = loader.load()

In [ ]:
# 문서 내용
print(docs1[10].page_content[:300])

In [ ]:
# 메타데이터
show_metadata(docs1)

PyPDF(OCR)

In [ ]:
loader2 = PyPDFLoader("https://arxiv.org/pdf/2103.15348.pdf", extract_images=True)  # 이미지 추출 옵션 활성화

docs2 = loader2.load()

In [ ]:
print(docs2[4].page_content[:300])  # 페이지 내용 접근

In [ ]:
show_metadata(docs2)

PyMuPDF

In [ ]:
loader3 = PyMuLoader(FILE_PATH)

docs3 = loader3.load()

In [ ]:
print(docs3[10].page_content[:300])  # 문서의 내용 출력

In [ ]:
show_metadata(docs3)

Unstructured

In [ ]:
loader4 = UnstructuredPDFLoader(FILE_PATH)

docs4 = loader4.load()

In [ ]:
print(docs4[0].page_content[:300])

In [ ]:
show_metadata(docs4)

In [ ]:
loader4_2 = UnstructuredPDFLoader(FILE_PATH, mode="elements")  # mode="elements": 텍스트 청크마다의 서로 다른 요소를 쉽게 분리

docs4_2 = loader4_2.load()

In [ ]:
print(docs4_2[0].page_content)

In [ ]:
set(doc4_2.metadata["category"] for doc4_2 in docs4_2)  # 데이터 카테고리 추출

In [ ]:
show_metadata(docs4_2)

PyPDFium2

In [ ]:
loader5 = PyPDFium2Loader(FILE_PATH)

docs5 = loader5.load()

In [ ]:
print(docs5[10].page_content[:300])

In [ ]:
show_metadata(docs5)

PDFMiner

In [ ]:
loader6 = PDFMinerLoader(FILE_PATH)

docs6 = loader6.load()

In [ ]:
print(docs6[0].page_content[:300])

In [ ]:
show_metadata(docs6)

In [ ]:
loader6_2 = PDFMinerPDFasHTMLLoader(FILE_PATH)

docs6_2 = loader6_2.load()

In [ ]:
print(docs6_2[0].page_content[:300])

In [ ]:
show_metadata(docs6_2)

In [ ]:
soup = BeautifulSoup(docs6_2[0].page_content, "html.parser")  # HTML 파서 초기화
content = soup.find_all("div")  # 모든 div 태그 검색

In [ ]:
cur_fs = None
cur_text = ""
snippets = []  # 동일한 글꼴 크기의 모든 스니펫 수집
for c in content:
    sp = c.find("span")
    if not sp:
        continue
    st = sp.get("style")
    if not st:
        continue
    fs = re.findall("font-size:(\d+)px", st)
    if not fs:
        continue
    fs = int(fs[0])
    if not cur_fs:
        cur_fs = fs
    if fs == cur_fs:
        cur_text += c.text
    else:
        snippets.append((cur_text, cur_fs))
        cur_fs = fs
        cur_text = c.text
snippets.append((cur_text, cur_fs))

In [ ]:
cur_idx = -1
semantic_snippets = []

# 제목 가정: 높은 글꼴 크기
for s in snippets:
    # 새 제목 판별: 현재 스니펫 글꼴 > 이전 제목 글꼴
    if (
        not semantic_snippets
        or s[1] > semantic_snippets[cur_idx].metadata["heading_font"]
    ):
        metadata = {"heading": s[0], "content_font": 0, "heading_font": s[1]}
        metadata.update(docs[0].metadata)
        semantic_snippets.append(Document(page_content="", metadata=metadata))
        cur_idx += 1
        continue

    # 동일 섹션 내용 판별: 현재 스니펫 글꼴 <= 이전 내용 글꼴
    if (
        not semantic_snippets[cur_idx].metadata["content_font"]
        or s[1] <= semantic_snippets[cur_idx].metadata["content_font"]
    ):
        semantic_snippets[cur_idx].page_content += s[0]
        semantic_snippets[cur_idx].metadata["content_font"] = max(
            s[1], semantic_snippets[cur_idx].metadata["content_font"]
        )
        continue

    # 새 섹션 생성 조건: 현재 스니펫 글꼴 > 이전 내용 글꼴, 이전 제목 글꼴 미만
    metadata = {"heading": s[0], "content_font": 0, "heading_font": s[1]}
    metadata.update(docs[0].metadata)
    semantic_snippets.append(Document(page_content="", metadata=metadata))
    cur_idx += 1

print(semantic_snippets[4])

PyPDF Directory

In [ ]:
loader7 = PyPDFDirectoryLoader("data/")

docs7 = loader7.load()

In [ ]:
len(docs7)

In [ ]:
print(docs7[50].page_content[:300])

In [ ]:
print(docs[50].metadata)

PDFPlumber

In [ ]:
loader8 = PDFPlumberLoader(FILE_PATH)

docs8 = loader8.load()

In [ ]:
print(docs8[10].page_content[:300])

In [ ]:
show_metadata(docs8)